# Course LLM Project

## LLM setup
Set up the langchain imports and settings

In [2]:
# Setup - Run this cell first
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent

from langchain_core.messages import AIMessage, ToolMessage
import re

# Use the exact model from the course
llm = ChatOllama(
    model="qwen2.5:7b",
    #temperature=0.0,      # deterministic for fair comparison
    temperature=0.7,      # more creative for better demonstration of prompt engineering
    num_ctx=8192,
)

print("Sucessfully loaded Qwen 2.5 7B Instruct via Ollama!")

import langchain
import langchain_ollama

print(langchain.__version__)
print(langchain_ollama.__version__)

Sucessfully loaded Qwen 2.5 7B Instruct via Ollama!
1.3.1
1.1.0


## Prompting (module 3)
Context: Answer queries that staff have regarding HR policies. While notifications are sent when policies are updated, many staff do not read the updated policy documents.


Develop prompting rules and constraints.

In [3]:
#simple_prompt = "Summarise the following document for a senior manager."

# Employee Benefits Policy
document = """Employee Benefits Policy – 2026

All full-time employees are eligible from their first day of employment for:

1. Comprehensive Medical Coverage: Includes outpatient, inpatient, and prescription drug coverage with annual limits. Dependents may be added at additional premium.

2. Dental and Vision Insurance: Preventive care fully covered; major dental work subject to annual caps. Vision coverage includes annual eye exams and subsidized glasses/lenses.

3. CPF Contribution: Company contributes 17% of basic salary to employee's Central Provident Fund. Employee contributes 20%. Employer top-up of up to SGD 1,500 annually for eligible employees.

4. Annual Wellness Stipend: SGD 500 per employee annually for gym memberships, wellness programs, or health-related activities.

5. Life Insurance: Company-paid term life insurance of 3x annual salary, automatically included for all full-time staff.

6. Disability Insurance: Long-term disability coverage protecting employees unable to work due to illness or injury.

All benefits are managed and tracked via the company's HR portal. Part-time employees receive pro-rata benefits based on contracted hours. Questions should be directed to the HR Benefits team at hr-benefits@company.sg."""

In [4]:
# Define a reusable ChatPromptTemplate with all 7 components
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are {role}.
        You help a Singapore-based enterprise manage their human resources.
        Always follow PDPA, company governance rules, and best practices for responsible AI.
        Always attempt to use tools, especially for temporal questions. 
        If tools and documentation that provide information for request are unavailable, explicitly state inability to answer due to lack of information and do not make up information.
    """),

    ("human", """### Context
        {context}

        ### User Instruction
        {user_instruction}

        ### Constraints
        {constraints}

        ### Output Format
        {output_format}

        ### Safety Instructions
        {safety_instructions}
    """)
])

# Create the LCEL chain
chain = prompt_template | llm | StrOutputParser()

query = "Summarise the document for an associate of the organisation. Focus only on their roles, responsibilities, entitlements, benefits, recommended next steps and impact of choices if any."
tools_list = ''

context_inputs = {
    "role": "Senior HR Officer",
    "context": document,
    "user_instruction": query,
    "tools_list": tools_list if tools_list else '',  # This should be defined based on the tools you have available. For example: "CurrentDateTool, EmployeeDatabaseTool, PolicyLookupTool"
    "constraints": 
        "You are engaging an associate of your organisation."
        "If any information is missing or not retrieveable or backed by any available sources, explicitly state \"Information not available\" and do not make up information.",
    
    "output_format":
        "You MUST explain where you retrieved the date from for any temporal questions."
        "Report which tool you used and what result it gave."
        "Then provide the 1 short paragraph answer based on the tool results."
        "Start the reply by echoing your understanding of the query, \"Regarding your query on...\""
        "Explain why you use each tool (or no tool) if they are available and how it helps you answer the question. If you cannot answer the question due to lack of information, explicitly say so and do not make up information.",
    
    "safety_instructions": "Ensure PDPA compliance. Do not disclose any personal data. Clearly flag any regulatory or compliance risks. Never suggest actions that could violate Singapore laws or regulations. If the document contains any sensitive information, do not include it in the summary and state \"Sensitive information redacted\". If request is an edge case or ambiguous, refer user to HR officer."
}

# Example usage with the same document
result = chain.invoke(context_inputs)

print("=== LANGCHAIN PROMPT TEMPLATE OUTPUT ===\n")
print(result)


=== LANGCHAIN PROMPT TEMPLATE OUTPUT ===

Regarding your query on summarizing the employee benefits policy for an associate of the organisation:

I understand you are seeking a summary that outlines roles, responsibilities, entitlements, benefits, recommended next steps, and the impact of choices. The information provided in the "Employee Benefits Policy – 2026" document does not include details about specific roles or responsibilities. Therefore, I will focus on entitlements, benefits, recommended next steps, and any potential impacts.

**Entitlements:**
- All full-time employees are eligible for comprehensive medical coverage, dental and vision insurance, CPF contributions, an annual wellness stipend, company-paid life insurance, and disability insurance.
- Part-time employees receive pro-rata benefits based on their contracted hours.

**Benefits:**
- Comprehensive Medical Coverage: Includes outpatient, inpatient, and prescription drug coverage with annual limits. Dependents may be a

## Test Query

In [5]:
query = "How long does Christmas last this year, and how much further is it from today?"

# Use the original context_inputs but override with the query
context_inputs["user_instruction"] = query
print("Context inputs\n")
print(context_inputs)

response_no_tool = chain.invoke(context_inputs)
print("=== LLM WITHOUT TOOLS (often wrong or hallucinated) ===\n")
print(response_no_tool)

Context inputs

{'role': 'Senior HR Officer', 'context': "Employee Benefits Policy – 2026\n\nAll full-time employees are eligible from their first day of employment for:\n\n1. Comprehensive Medical Coverage: Includes outpatient, inpatient, and prescription drug coverage with annual limits. Dependents may be added at additional premium.\n\n2. Dental and Vision Insurance: Preventive care fully covered; major dental work subject to annual caps. Vision coverage includes annual eye exams and subsidized glasses/lenses.\n\n3. CPF Contribution: Company contributes 17% of basic salary to employee's Central Provident Fund. Employee contributes 20%. Employer top-up of up to SGD 1,500 annually for eligible employees.\n\n4. Annual Wellness Stipend: SGD 500 per employee annually for gym memberships, wellness programs, or health-related activities.\n\n5. Life Insurance: Company-paid term life insurance of 3x annual salary, automatically included for all full-time staff.\n\n6. Disability Insurance: Lo

## Build Tools for LLM

In [6]:
from langchain_core.tools import tool
from zoneinfo import ZoneInfo
from datetime import datetime
import requests

@tool
def calculator(expression: str) -> str:
    """
    - Purpose: Evaluate a mathematical expression and return the numeric result.

    - Inputs:
    expression (string, REQUIRED): A valid mathematical expression (e.g. "25 - 6", "10 * 3 + 2")

    - Output:
    result (string): The evaluated numeric result as a string

    - Example:
    Input: {"expression": "25 - 6"}
    Output: "19"

    - Notes:
    Use this tool whenever numerical computation, time difference, or arithmetic is required.
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_singapore_date() -> str:
    """
    - Purpose: Get the current date in Singapore (UTC+8 timezone).

    - Inputs:
    None

    - Output:
    date (string): Current date in format "YYYY-MM-DD Singapore Date"

    - Example:
    Output: "2026-06-14 Singapore Date"

    - Notes:
    This tool MUST be used for any question involving current date, duration, or temporal comparison.
    """
    now = datetime.now(ZoneInfo("Asia/Singapore"))
    return now.strftime("%Y-%m-%d Singapore Date")

@tool
def get_current_singapore_time() -> str:
    """
    - Purpose: Get the current time in Singapore (UTC+8 timezone).

    - Inputs:
    None

    - Output:
    time (string): Current time in format "HH:MM:SS Singapore Time"

    - Example:
    Output: "14:32:10 Singapore Time"

    - Notes:
    Use for any question involving current time, deadlines, or time-based comparisons.
    """
    now = datetime.now(ZoneInfo("Asia/Singapore"))
    return now.strftime("%H:%M:%S Singapore Time")

@tool
def get_exchange_rate(base: str, target: str) -> str:
    """
    - Purpose: Retrieve exchange rate conversion between currencies relative to SGD.

    - Inputs:
    base (string, REQUIRED): Base currency (e.g. "SGD", "USD")
    target (string, REQUIRED): Target currency (e.g. "USD", "EUR", "JPY", "MYR")

    - Output:
    rate (string): Conversion rate in format "1 BASE = X TARGET"

    - Example:
    Input: {"base": "SGD", "target": "USD"}
    Output: "1 SGD = 0.74 USD"

    - Notes:
    Use this tool whenever currency conversion or financial comparison is required.
    """
    rates = {"USD": 0.74, "EUR": 0.68, "JPY": 110.5, "MYR": 3.45}
    rate = rates.get(target.upper(), "Unknown currency")
    return f"1 {base.upper()} = {rate} {target.upper()}"

@tool
def safety_compliance_checker(text: str) -> str:
    """
    safety_compliance_checker:
    - Purpose: Check user input for PDPA, security, or policy violations.

    - Inputs:
    text (string, REQUIRED): User instruction or message to analyze

    - Output:
    result (string): Either "No obvious compliance breaches." or a list of detected risks

    - Example:
    Input: {"text": "send email to test@example.com"}
    Output: "PDPA Warning(s): Possible email address."

    - Notes:
    Always run this tool when user input contains sensitive, risky, or ambiguous instructions.
    """
    findings = []

    if "@" in text:
        findings.append("Possible email address.")

    if "{" in text and "}" in text:
        findings.append("Possible malicious injections.")

    if "danger" in text:
        findings.append("Possible dangerous search.")

    if not findings:
        return "No obvious compliance breaches."

    return "PDPA Warning(s): " + ", ".join(findings)

@tool
def public_holiday_checker(holiday: str, year: str) -> str:
    """public_holiday_checker:
    - Purpose: Get official Singapore public holiday date and duration, only between 2024 and 2026.

    - Inputs:
    - holiday (string, REQUIRED): Name of holiday (e.g. "Christmas", "New Year", "Labour Day")
    - year (string, REQUIRED): Year of interest (e.g. 2026)

    - Output:
    - date (YYYY-MM-DD)
    - day of week
    - duration in days

    - Example:
    Input: {"holiday": "Christmas", "year": "2026"}
    Output: {"date": "2026-12-25", "duration_days": 1}
    """
    # Todo, fetch from online API or database in real implementation
    holidays = {
        '2026': {
            'New Year’s Day': {'date': '2026-01-01', 'day': 'Thursday', 'duration': 1},
            'Chinese New Year': {'date': '2026-02-17', 'day': 'Tuesday', 'duration': 2},
            'Hari Raya Puasa': {'date': '2026-03-21', 'day': 'Saturday', 'duration': 1},
            'Good Friday': {'date': '2026-04-03', 'day': 'Friday', 'duration': 1},
            'Labour Day': {'date': '2026-05-01', 'day': 'Friday', 'duration': 1},
            'Hari Raya Haji': {'date': '2026-05-27', 'day': 'Wednesday', 'duration': 1},
            'Vesak Day': {'date': '2026-05-31', 'day': 'Sunday', 'duration': 1},
            'National Day': {'date': '2026-08-09', 'day': 'Sunday', 'duration': 1},
            'Deepavali': {'date': '2026-11-08', 'day': 'Sunday', 'duration': 1},
            'Christmas': {'date': '2026-12-25', 'day': 'Friday', 'duration': 1}
        },
        '2025': {
            'New Year’s Day': {'date': '2025-01-01', 'day': 'Wednesday', 'duration': 1},
            'Chinese New Year': {'date': '2025-01-29', 'day': 'Wednesday', 'duration': 2},
            'Hari Raya Puasa': {'date': '2025-02-17', 'day': 'Monday', 'duration': 1},
            'Good Friday': {'date': '2025-02-28', 'day': 'Friday', 'duration': 1},
            'Labour Day': {'date': '2025-03-03', 'day': 'Monday', 'duration': 1},
            'Hari Raya Haji': {'date': '2025-03-14', 'day': 'Friday', 'duration': 1},
            'Vesak Day': {'date': '2025-03-25', 'day': 'Tuesday', 'duration': 1},
            'National Day': {'date': '2025-08-09', 'day': 'Saturday', 'duration': 1},
            'Deepavali': {'date': '2025-10-29', 'day': 'Wednesday', 'duration': 1},
            'Christmas': {'date': '2025-12-25', 'day': 'Thursday', 'duration': 1}
        },
        '2024': {
            'New Year’s Day': {'date': '2024-01-01', 'day': 'Monday', 'duration': 1},
            'Chinese New Year': {'date': '2024-02-12', 'day': 'Monday', 'duration': 2},
            'Hari Raya Puasa': {'date': '2024-02-22', 'day': 'Thursday', 'duration': 1},
            'Good Friday': {'date': '2024-02-29', 'day': 'Thursday', 'duration': 1},
            'Labour Day': {'date': '2024-03-01', 'day': 'Friday', 'duration': 1},
            'Hari Raya Haji': {'date': '2024-03-11', 'day': 'Monday', 'duration': 1},
            'Vesak Day': {'date': '2024-03-25', 'day': 'Monday', 'duration': 1},
            'National Day': {'date': '2024-08-09', 'day': 'Friday', 'duration': 1},
            'Deepavali': {'date': '2024-10-01', 'day': 'Tuesday', 'duration': 1},
            'Christmas': {'date': '2024-12-25', 'day': 'Wednesday', 'duration': 1}
        }
    }

    all_years = list(holidays.keys())
    all_holidays = [holiday for year_data in holidays.values()
                    for holiday in year_data.keys()]

    if holiday in all_holidays and year in all_years:
        return f"{holiday} is a {holidays[year][holiday]['duration']} day(s) public holiday in Singapore, starting on {holidays[year][holiday]['date']} which is a {holidays[year][holiday]['day']}."
    elif year not in all_years:
        return f"Data for the specific year {year} is not available."
    elif holiday not in all_holidays:
        return f"{holiday} is not a recognised public holiday in Singapore."

@tool
def days_between_dates_calculator(start_date: str, end_date: str) -> str:
    """
    Calculate number of days between two dates. End date must be later than start date.

    Inputs:
    - start_date: YYYY-MM-DD
    - end_date: YYYY-MM-DD

    Output:
    - Integer day difference
    """

    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    delta = (end - start).days

    return str(delta)

tools = [calculator, get_current_singapore_date, get_current_singapore_time, get_exchange_rate, safety_compliance_checker, public_holiday_checker, days_between_dates_calculator]
print("Tools defined:")
for t in tools:
    #print(f"- {t.name}: {t.description}")

    print(t.name)
    print(t.args)

Tools defined:
calculator
{'expression': {'title': 'Expression', 'type': 'string'}}
get_current_singapore_date
{}
get_current_singapore_time
{}
get_exchange_rate
{'base': {'title': 'Base', 'type': 'string'}, 'target': {'title': 'Target', 'type': 'string'}}
safety_compliance_checker
{'text': {'title': 'Text', 'type': 'string'}}
public_holiday_checker
{'holiday': {'title': 'Holiday', 'type': 'string'}, 'year': {'title': 'Year', 'type': 'string'}}
days_between_dates_calculator
{'start_date': {'title': 'Start Date', 'type': 'string'}, 'end_date': {'title': 'End Date', 'type': 'string'}}


In [19]:
#You MUST decompose every query into ALL necessary tool calls.
#If a task requires multiple steps, you MUST output multiple tool calls, and order them.
#Do not stop after the first tool if more tools are needed to complete the task.
#Always plan the full tool chain in one response.
#If subsequent tools require the result of previous tool, use a placeholder ($variable_name) to signify the result of the preceding tool call. Do not invent variable values or simulate results.


tool_call_prompt_template = prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
        You are a tool planner.

        Your ONLY job is to select which tools to call.

        You MUST NOT:
        - answer the question
        - explain reasoning
        - simulate results
        - execute tools
     
        Based on user query and past tool calls, you are ONLY allowed to choose ONE best next action.
        Return exactly one tool call OR final answer.
        Do not plan future steps.
        Do not assume future tools.
        Return tool calls using the system format.
        If required information is already present in tool results, return NO tool calls.

        If the information already available in Tool State is sufficient to answer the user's question, return NO_TOOL.

        Never call the same tool again if that tool has already been used, inputs required are the same, and its output already contains all relevant information.

        Only call a tool again if:
        - different inputs are required, or
        - another tool depends on it.

        Repeated tool calls with identical arguments are forbidden.

        If the query depends on the CURRENT date, obtain current date first using tool.
        If the query specifies an explicit year/date, do not call current date tools unless required.
        
        YOU ARE STRICTLY FORBIDDEN FROM GUESSING OR INFERRING ANY OF THE FOLLOWING:
        - years
        - dates
        - current time
        - holidays
        - durations
        - numeric values

        You MUST use tools for:
        - arithmetic
        - date differences
        - duration calculations
        Never perform calculations yourself.

        If any such value is required and not explicitly provided by the user or tools:
        YOU MUST use a placeholder.
        Examples:
        - $current_year
        - $current_date
        - $holiday_year
        NEVER output a real DATE OR TIME unless it came directly from a tool or user input.
     
        ### Absolutes
        MANDATORY FOR TEMPORAL QUESTIONS:
        1. Use current date and time if not provided in the query or context.
        2. You NEVER know today's date, time, or year from your training data, context or memory.
        3. For ANY question involving time, dates, years, durations, or temporal comparisons: ALWAYS FIRST call tools that establish the current date. 
        4. NEVER derive any current date and time information from the document context - that is historical data, not the current date.
        5. Only after getting the current date or time should you proceed with other tools.
        6. If you cannot determine the current date or time via tools, explicitly say you cannot answer the temporal question.
            
        ### Available Tools
        You have access to these tools: {tools_list} 
     """),

    ("human", """
        ### User Instruction
        {user_instruction}
    """),

    ("tool", """
        ### Tool State
        {state}, where
     
        state is a dictionary of the following structure:
            "memory": dict           # results from prior tool calls rtto form memory of LLM 
            "history": list          # full ordered trace of tool calls inputs given and outputs received, check this for past tools planned for future planning and termination of plans
            "executed": set,         # deduplication guard for manual checking 
            "step": int              # code tracks number of looped llm queries
     
        You MUST NOT infer missing values not present in this state.
    """)
])

tool_call_context_inputs = {
    "tools_list": tools_list if tools_list else '',  # This should be defined based on the tools you have available. For example: "CurrentDateTool, EmployeeDatabaseTool, PolicyLookupTool"
    "user_instruction": query,
}

ValueError: Unexpected message type: tool. Use one of 'human', 'user', 'ai', 'assistant', or 'system'.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

# Create the LCEL chain
chain_with_tools = tool_call_prompt_template | llm_with_tools #| StrOutputParser()

context_inputs["user_instruction"] = query
context_inputs["tools_list"] = ", ".join([f"- {tool.name}: {tool.description}\n" for tool in tools])

#response = chain_with_tools.invoke(context_inputs)

# # DEBUG: See the full response
# print("=== FULL RESPONSE OBJECT ===\n")
# print(f"Response type: {type(response)}")
# print(f"Response: {response}\n")

# # Check all attributes
# if hasattr(response, "__dict__"):
#     print(f"Response attributes: {response.__dict__}\n")

# # Try to access content
# if hasattr(response, "content"):
#     print(f"Response content: {response.content}\n")

# # Now get tool calls
# tool_calls = response.tool_calls if hasattr(response, "tool_calls") else []
# print("=== RAW TOOL CALLS GENERATED BY LLM ===\n")
# print(f"Tool calls: {tool_calls}")

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
import json

def resolve_placeholders(args, state):
    """
    Replace $variables in tool args using state.
    """

    def resolve_value(v):
        if isinstance(v, str) and v.startswith("$"):
            key = v[1:]
            if key not in state:
                raise ValueError(f"Missing dependency: {key}")
            return state[key]
        return v

    return {k: resolve_value(v) for k, v in args.items()}

def tool_generation_loop(context_inputs):

    state = {
    #"context": {},          # derived values for LLM reasoning
    "memory": {},           # persistent tool outputs (key-value)
    "history": [],          # full trace of tool calls
    "executed": set(),      # deduplication guard
    "step": 0               # loop control
    }

    max_steps = 10

    while True:
        state["step"] += 1
        if state["step"] > max_steps:
            raise Exception("Too many tool iterations")

        context_inputs["state"] = state["memory"]
        response = chain_with_tools.invoke(context_inputs)
        tool_calls = response.tool_calls if hasattr(response, "tool_calls") else []
        print(tool_calls)

        # If no tool calls, we are done
        if not tool_calls:
            break
            
        tc = tool_calls[0]  # Get the first tool call
        tool_name = tc["name"]
        args = tc["args"]

        # build execution signature
        signature = f"{tool_name}:{json.dumps(args, sort_keys=True)}"

        # dedup check
        if signature in state["executed"]:
            print("Skipping duplicate tool call:", signature)
            break

        state["executed"].add(signature)

         # resolve placeholders
        resolved_args = resolve_placeholders(args, state["memory"])

        # run tool
        tool = next(t for t in tools if t.name == tool_name)
        result = tool.invoke(resolved_args)

        # store in history (FULL TRACE)
        state["history"].append({
            "tool": tool_name,
            "args": args,
            "resolved_args": resolved_args,
            "result": result
        })

        # store in memory (LLM-facing facts)
        state["memory"][tool_name] = result

        # optional derived fields
        if tool_name == "get_current_singapore_date":
            state["memory"]["current_date"] = result
            state["memory"]["current_year"] = result[:4]
            state["memory"]["current_month"] = result[5:7]
            state["memory"]["current_day"] = result[8:10]

    return state

# Simple chain: LLM → Tools → LLM
# tool_results, tool_state = execute_tools(tool_calls)
tool_state = tool_generation_loop(context_inputs)

# for tool_call in plan:
#     result = run_tool(tool_call)
    # store(result)

print("=== TOOL RESULTS ===\n")
print(tool_state)
print("\n")


# tool_execution_context = {
#     #"tool_calls": tool_calls,
#     #"tool_results": tool_results,
#     "tool_state": tool_state
# }

[{'name': 'get_current_singapore_date', 'args': {}, 'id': '4b4c2d88-bb73-4e6a-a00b-d9b3ddbd5e29', 'type': 'tool_call'}]
[{'name': 'public_holiday_checker', 'args': {'holiday': 'Christmas', 'year': '2026'}, 'id': 'b4901d17-c838-48f5-84d0-67a92670a2f2', 'type': 'tool_call'}]
[{'name': 'public_holiday_checker', 'args': {'holiday': 'Christmas', 'year': '2026'}, 'id': 'e9eb409c-b104-4527-bf1d-030567492721', 'type': 'tool_call'}]
Skipping duplicate tool call: public_holiday_checker:{"holiday": "Christmas", "year": "2026"}
=== TOOL RESULTS ===

{'memory': {'get_current_singapore_date': '2026-06-24 Singapore Date', 'current_date': '2026-06-24 Singapore Date', 'current_year': '2026', 'current_month': '06', 'current_day': '24', 'public_holiday_checker': 'Christmas is a 1 day(s) public holiday in Singapore, starting on 2026-12-25 which is a Friday.'}, 'history': [{'tool': 'get_current_singapore_date', 'args': {}, 'resolved_args': {}, 'result': '2026-06-24 Singapore Date'}, {'tool': 'public_holida

In [ ]:
# Feed results back to LLM using message history
# Add final request for answer - use base LLM without tools to get text response
#final_prompt_text = f"Based on the tool results above, please provide a clear and concise answer to the user's question. {query}"
 
messages = [
    HumanMessage(content= query),
    #ToolMessage(content="", tool_calls=tool_state["memory"], tool_call_id="0"),  # Initial tool calls from LLM

    HumanMessage(content="Based on the tool results below, please provide a clear and concise answer to the user's question.")
]

for i, tool_call in enumerate(tool_state["memory"].items()):
    tool_name, tool_result = tool_call
    messages.append(ToolMessage(content=tool_result, tool_calls=[], tool_call_id=str(i+1)))

final_response = llm.invoke(messages)

print("=== FINAL ANSWER AFTER TOOL USE ===\n")
print(final_response.content)
print("\n")


=== FINAL ANSWER AFTER TOOL USE ===

Christmas this year (2026) will last for 1 day, starting on December 25. From today, the number of days until Christmas depends on the current date. If today's date is June 24, 2026, then there are approximately 183 days remaining until Christmas.




## Module 10

In [ ]:
# === SETUP (run first) ===
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Import LANGSMITH_API_KEY
from pathlib import Path

secret_file = Path("secret.txt")

with open(secret_file, "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        os.environ[key] = value
            
# === LANG SMITH MONITORING ENABLED ===
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
#os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "NUS_LLMP_Evaluation_Exercise"

llm = ChatOllama(model="llama3.1:8b", temperature=0.0, num_ctx=8192)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("Setup complete – Ollama + LLaMA 3.1 8B + ChromaDB + Local Embeddings ready!")

Setup complete – Ollama + LLaMA 3.1 8B + ChromaDB + Local Embeddings ready!


In [ ]:
# Four realistic policy documents for RAG demonstration
docs = [
    Document(page_content="""Annual Leaves for Full-Time Employees
All full-time staff are entitied to annual leave (20-30 days based on tenure, increasing by 2 every 5 years). Leave requests are to be made 2 weeks in advance, and a maximum of 5 unused days are allowed to carry over to the next year.""", metadata={"source":"hr_annual_leaves.pdf", "department":"HR", "date":"2026-03"}),

    Document(page_content="""Annual Leaves for Part-Time Employees
Part-time employees are entitled to annual leave on a pro-rata basis equivalent to their contracted hours compared to a full-time schedule, with all requests requiring manager approval via the HR portal at least two weeks prior to the intended leave date to ensure adequate departmental coverage.""", metadata={"source":"hr_annual_leaves.pdf", "department":"HR", "date":"2026-03"}),

    Document(page_content="""Employee Benefits
All full-time employees are eligible on their first day of employment for comprehensive medical, dental, and vision insurance, CPF contribution, and a $500 annual wellness stipend, all managed and tracked via the company's HR portal.""", metadata={"source":"hr_benefits.pdf", "department":"HR", "date":"2026-02"}),

    Document(page_content="""Whistleblowing
Employees are guaranteed absolute confidentiality and strict protection from retaliation when reporting suspected illegal, unethical, or fraudulent activities through the company's secure compliance hotline, which automatically initiates an immediate and independent investigation by the HR ethics committee.""", metadata={"source":"whistleblowing.pdf", "department":"Security", "date":"2026-01"}),

    Document(page_content="""Maintenance Inspection Protocol
To maximize facility uptime and ensure safety, all manufacturing machinery must undergo a mandatory 50-point preventative maintenance inspection every 10,000 operating hours, with certified technicians logging real-time telemetry into the Asset Management platform before the production line can be re-authorized for use.""", metadata={"source":"maintenance_protocol.pdf", "department":"Operations", "date":"2026-04"}),

    Document(page_content="""Travel Claims
All employees traveling on approved corporate business must submit itemized receipts for pre-authorized lodging, transport, and meal expenses through the expense portal within 30 days of travel completion to receive reimbursement, subject to standard daily per diem limits and mandatory manager approval.""", metadata={"source":"hr_travel_claims.pdf", "department":"HR", "date":"2026-04"}),

    Document(page_content="""Usage of AI and LLMs
Employees must obtain written authorization from the IT Security Department before using company-approved generative AI or LLM tools, are strictly prohibited from inputting proprietary, confidential, or customer-sensitive data into these platforms, and remain fully accountable for verifying the accuracy and ethical compliance of all AI-generated outputs used in professional deliverables.""", metadata={"source":"hr_ai_llm_use.pdf", "department":"HR", "date":"2026-04"}),

    Document(page_content="""Performance Management and Appraisal System
All employees undergo annual performance reviews conducted by their direct manager using the standardized 5-point rating scale. Reviews are scheduled in June and December, with feedback sessions required within one month. Performance ratings influence salary increments, promotions, and professional development opportunities. Employees may appeal their ratings through the formal grievance process.""", metadata={"source":"hr_performance_management.pdf", "department":"HR", "date":"2026-05"}),

    Document(page_content="""Code of Conduct
All employees must adhere to the company's Code of Conduct which prohibits discrimination, harassment, and unethical behavior. Violations including conflicts of interest, breach of confidentiality, or inappropriate use of company resources will result in disciplinary action ranging from warnings to termination depending on severity. All staff are required to complete mandatory conduct training within their first month.""", metadata={"source":"hr_code_of_conduct.pdf", "department":"HR", "date":"2026-02"}),

    Document(page_content="""Professional Development and Training
The company allocates SGD 2,000 per employee annually for approved professional development, including courses, certifications, and conferences. Employees must obtain manager and HR approval for training exceeding SGD 1,000. All training requests must align with organizational strategic objectives. Employees receiving training support are required to remain with the company for a minimum of 12 months post-training.""", metadata={"source":"hr_professional_development.pdf", "department":"HR", "date":"2026-04"}),

    Document(page_content="""Compensation and Salary Structure
Salaries are benchmarked quarterly against industry standards and adjusted annually based on performance reviews, market conditions, and cost of living increases. The company provides transparent grade-to-salary band mappings accessible through the HR portal. Discretionary bonuses of up to 4 months are allocated based on individual and company performance, typically distributed in Q1 and Q4.""", metadata={"source":"hr_compensation.pdf", "department":"HR", "date":"2026-03"}),

    Document(page_content="""Workplace Safety and Health
All employees are responsible for maintaining a safe work environment and must report hazards immediately to their manager and the Health and Safety Committee. The company provides annual health and safety training, ergonomic assessments, and mental health support through the Employee Assistance Programme. Accidents and near-misses must be documented within 24 hours via the Safety Incident Portal.""", metadata={"source":"hr_safety_health.pdf", "department":"HR", "date":"2026-04"}),

    Document(page_content="""Flexible Work Arrangements and Remote Work Policy
Eligible employees may request flexible working hours or remote work arrangements subject to manager approval and operational requirements. Remote work is permitted up to 2 days per week, with core working hours between 10 AM and 3 PM Singapore time. All remote workers must maintain confidentiality, use secure VPN connections, and ensure proper data security. Annual in-person team reviews are mandatory.""", metadata={"source":"hr_flexible_work.pdf", "department":"HR", "date":"2026-05"}),

    Document(page_content="""Grievance and Dispute Resolution
Employees may formally lodge grievances through the HR department regarding pay, working conditions, treatment, or contractual disputes. All grievances are investigated within 14 days of submission by an impartial committee. Employees have the right to representation and confidentiality is maintained throughout the process. Appeals may be escalated to senior management if not resolved satisfactorily.""", metadata={"source":"hr_grievance_resolution.pdf", "department":"HR", "date":"2026-03"}),

    Document(page_content="""Attendance and Punctuality Policy
Employees are expected to maintain regular attendance and punctuality as per their contracted hours. Unexcused absences exceeding 2 consecutive days may result in disciplinary action. All absences must be reported to the manager before the start of work. Excessive late arrivals (more than 3 times in a month) will trigger a formal performance discussion with HR.""", metadata={"source":"hr_attendance.pdf", "department":"HR", "date":"2026-02"}),

    Document(page_content="""Dress Code and Professional Appearance
The company maintains a business casual dress code in most departments, with specific requirements for client-facing roles. Smart casual attire is expected; jeans, sandals, and graphic t-shirts are not permitted. Department heads may establish stricter guidelines based on client interactions. Violations of the dress code will be addressed through informal counseling, with repeated breaches escalated to formal warnings.""", metadata={"source":"hr_dress_code.pdf", "department":"HR", "date":"2026-01"}),

    Document(page_content="""Maternity, Paternity, and Parental Leave
Female employees are entitled to 8 weeks of paid maternity leave (4 weeks before and 4 weeks after delivery) plus 4 weeks of unpaid leave under the Employment Act. Male employees receive 4 weeks of paid paternity leave. Adoption leave of 4 weeks is available for adopting employees. All leave must be coordinated with HR and the employee's manager at least 8 weeks prior to the expected leave date.""", metadata={"source":"hr_parental_leave.pdf", "department":"HR", "date":"2026-04"}),

    Document(page_content="""Medical and Compassionate Leave
Employees are entitled to 14 days of paid medical leave annually for personal illness or caring for immediate family members. Medical certificates are required for absences exceeding 2 consecutive days. Compassionate leave of up to 3 days is provided for bereavement of immediate family members. All medical leave requests must be reported to HR within 24 hours.""", metadata={"source":"hr_medical_leave.pdf", "department":"HR", "date":"2026-03"})
]

print(f"Loaded {len(docs)} substantive documents for RAG demonstration.")

Loaded 18 substantive documents for RAG demonstration.


In [ ]:
# Step 1–4: Loading → Splitting → Embedding → Vector Store
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="hr_policies_rag",
    persist_directory="./chroma_db"
)

# The following retriever will be used in the next step for RAG question-answering. It retrieves the top 3 most relevant chunks based on cosine similarity.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Indexed {len(splits)} chunks into ChromaDB vector store.")

Indexed 18 chunks into ChromaDB vector store.


In [ ]:
query = "According to our policies, how can employees deal with travel claims?"

no_rag_prompt = ChatPromptTemplate.from_template("Answer the question:\n\nQuestion: {q}")
no_rag_chain = no_rag_prompt | llm | StrOutputParser()

response_no_rag = no_rag_chain.invoke({"q": query})
print("=== WITHOUT RAG ===\n")
print(response_no_rag)    

=== WITHOUT RAG ===

According to standard HR policies, employees typically follow these steps when dealing with travel claims:

1. **Submit a claim within the specified timeframe**: Employees usually have a certain number of days (e.g., 30 or 60) from the date of return to submit their travel claim.
2. **Gather required documentation**: This may include receipts for flights, accommodations, meals, and other expenses related to the trip, as well as any supporting documents such as boarding passes, hotel bills, or invoices.
3. **Complete a claim form**: Employees will typically need to fill out a standard claim form provided by their employer, which will ask for details about the trip, including dates, destinations, and expenses incurred.
4. **Attach supporting documentation**: The completed claim form should be accompanied by all relevant receipts and documents to support the claimed expenses.
5. **Submit the claim**: Employees submit the completed claim form and attached documentation

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([ 
    ("system", 
    """ 
        You are a document retrieval system. 
        You must answer ONLY using information explicitly present in the supplied context. 
        
        Rules: 
        - Do not use outside knowledge. 
        - Do not guess. 
        - Do not infer. 
        - Do not combine facts from memory. 
        - Quote relevant facts from the context where possible. 
        
        If the answer is not fully contained in the context, reply exactly: NO_RELEVANT_INFORMATION 
    """), 
    ("human", 
    """ 
        Context: {context} 
        Question: {q} 
    """) ])

def format_docs(docs_list):
    return "\n\n---\n\n".join(doc.page_content for doc in docs_list)

rag_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

@tool 
def hr_policy_retriever(query): 
    """ 
    Search internal HR documents and policies.

    MUST use this tool whenever the user asks about:

    - annual leave
    - sick leave
    - childcare leave
    - maternity leave
    - benefits
    - company rules
    - employee handbook
    - HR policies
    - internal procedures
    - entitlements

    This tool contains the authoritative source of truth.

    Do not answer these questions without using this tool.

    Input:
    query (string)

    Output:
    HR policy information from internal documents.
    """

    seen = set()

    retrieved_docs = retriever.invoke(query) 
    context = format_docs(retrieved_docs)

    state = { 
        "query": query, # original user query 
        "sources": [], # persistent tool outputs (key-value) 
        "retrieved_docs": {}, # full trace of retrieved documents 
        #"rag_answer": ""  # content of the retrieved document 
    } 

    #answer = rag_chain.invoke({
    #    "context": context,
    #    "q": query
    #})

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "unknown")

        if source not in seen:
            seen.add(source)
            state["sources"].append(source)
            state["retrieved_docs"][source] = doc.page_content  # store a snippet of the doc content for reference 
    
    #state["rag_answer"] = answer 

    return state 

tools.append(hr_policy_retriever)
llm_with_tools = llm.bind_tools(tools)
chain_with_tools = tool_call_prompt_template | llm_with_tools #| StrOutputParser()

In [ ]:
# Inspect the top retrieved documents for the query to understand what context the model had access to when generating the RAG response.
# How many documents were retrieved, and what information did they contain? This can help explain the model's answer and identify any gaps in the retrieved context.
# Which one of the fours documents was not retrived and why?

retrieved_docs = retriever.invoke(query)
print("=== RETRIEVED CONTEXT ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1} (source: {doc.metadata['source']}):\n{doc.page_content}\n---\n")

=== RETRIEVED CONTEXT ===

Document 1 (source: hr_travel_claims.pdf):
Travel Claims
All employees traveling on approved corporate business must submit itemized receipts for pre-authorized lodging, transport, and meal expenses through the expense portal within 30 days of travel completion to receive reimbursement, subject to standard daily per diem limits and mandatory manager approval.
---

Document 2 (source: hr_travel_claims.pdf):
Travel Claims
All employees traveling on approved corporate business must submit itemized receipts for pre-authorized lodging, transport, and meal expenses through the expense portal within 30 days of travel completion to receive reimbursement, subject to standard daily per diem limits and mandatory manager approval.
---

Document 3 (source: hr_travel_claims.pdf):
Travel Claims
All employees traveling on approved corporate business must submit itemized receipts for pre-authorized lodging, transport, and meal expenses through the expense portal within 30 day

In [ ]:
query = "According to our policies, compare the differences in annual leaves between full-time and part-time staff."
no_rag_prompt = ChatPromptTemplate.from_template("Answer the question:\n\nQuestion: {q}")
no_rag_chain = no_rag_prompt | llm | StrOutputParser()

response_no_rag = no_rag_chain.invoke({"q": query})
print("=== WITHOUT RAG ===\n")
print(response_no_rag)

=== WITHOUT RAG ===

Based on general HR policies, here's a comparison of the differences in annual leave entitlements between full-time and part-time staff:

**Full-Time Staff:**

* Typically entitled to a minimum number of paid annual leave days per year (e.g., 20-25 days)
* May be eligible for additional leave days based on length of service or performance
* Often have more flexibility in taking leave, including the ability to take longer periods off with pay

**Part-Time Staff:**

* Entitled to a proportionate number of paid annual leave days per year, based on their percentage of full-time hours worked (e.g., 10-15 days for a part-timer working 50% of full-time hours)
* May have restrictions on taking leave during peak periods or when their role requires coverage
* Often have less flexibility in taking leave, with more limited options for longer periods off

**Key differences:**

1. **Leave entitlement:** Full-time staff typically receive more annual leave days than part-time staf

In [ ]:
# Incorporate rag as tool
context_inputs["user_instruction"] = query
context_inputs["tools_list"] = ", ".join([f"- {tool.name}: {tool.description}\n" for tool in tools])

tool_state = tool_generation_loop(context_inputs)

print("=== TOOL RESULTS ===\n")
print(tool_state)
print("\n")

messages = [
    HumanMessage(content= query),
    #ToolMessage(content="", tool_calls=tool_state["memory"], tool_call_id="0"),  # Initial tool calls from LLM

    ToolMessage(content=
    """
    Based on the tool results below, please provide a clear and concise answer to the user's question.
    If there is no tools, answer to the best of your ability based on your knowledge and the context provided.
    Do not make up any new concrete information.
    """, 
    tool_call_id=str(1))
]

for i, tool_call in enumerate(tool_state["memory"].items()):
    tool_name, tool_result = tool_call
    messages.append(ToolMessage(content=tool_result, tool_calls=[], tool_call_id=str(i+2)))

final_response = llm.invoke(messages)
    
print("=== FINAL ANSWER AFTER TOOL USE ===\n")
print(final_response.content)
print("\n")

[{'name': 'hr_policy_retriever', 'args': {'query': 'annual leave policy comparison between full-time and part-time staff'}, 'id': 'ce6847bf-4828-43bd-9058-9c97df30a9e3', 'type': 'tool_call'}]
[{'name': 'hr_policy_retriever', 'args': {'query': 'annual leave policy comparison between full-time and part-time staff'}, 'id': 'e1d8f34f-da18-408a-9a46-76e53c906c19', 'type': 'tool_call'}]
Skipping duplicate tool call: hr_policy_retriever:{"query": "annual leave policy comparison between full-time and part-time staff"}
=== TOOL RESULTS ===

{'memory': {'hr_policy_retriever': {'query': 'annual leave policy comparison between full-time and part-time staff', 'sources': ['hr_annual_leaves.pdf'], 'retrieved_docs': {'hr_annual_leaves.pdf': 'Annual Leaves for Full-Time Employees\nAll full-time staff are entitied to annual leave (20-30 days based on tenure, increasing by 2 every 5 years). Leave requests are to be made 2 weeks in advance, and a maximum of 5 unused days are allowed to carry over to the n